# Summary of Trigger Proposal

In [1]:
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

In [2]:
import ocha_stratus as stratus
from src.datasources.era5 import fetch_era5_data
from src.datasources.seas5 import fetch_seas5_data
from src.constants import *
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, ScalarFormatter
from affine import Affine
from rasterstats import zonal_stats
import calendar
import math

In [3]:
pd.options.display.float_format = "{:,.0f}".format

In [4]:
mam_blob_name = "ds-aa-eth-drought/exploration/Ethiopia MAM zones.csv"
start_date = pd.Timestamp("1997-01-01")
end_date = pd.Timestamp("2025-12-01")
season_months = [3, 4, 5]

In [5]:
# reading in population data
eth_adm_pop = stratus.load_csv_from_blob(
    "ds-aa-eth-drought/processed/eth_adm2_pop.csv",
    stage="dev",
    container_name="projects",
)
mam_validation_data = (
    "ds-aa-eth-drought/exploration/Ethiopia Drought Validation Data MAM.csv"
)
validation_csv = stratus.load_csv_from_blob(
    mam_validation_data, stage="dev", container_name="projects"
)
validation_csv["year"] = validation_csv["Season"].str[-4:].astype(int)

In [6]:
season_csv = stratus.load_csv_from_blob(
    mam_blob_name, stage="dev", container_name="projects"
)
seas5_data = fetch_seas5_data(
    season_csv["adm2_pcode"].unique(), start_date=start_date, end_date=end_date
)

era5_data = fetch_era5_data(
    season_csv["adm2_pcode"].unique(), start_date=start_date, end_date=end_date
)
era5_data["year"] = pd.to_datetime(era5_data["valid_date"]).dt.year
era5_data["valid_month"] = pd.to_datetime(era5_data["valid_date"]).dt.month
era5_data["mean"] = era5_data["mean"] * era5_data["valid_month"].map(days_in_month)
era5_df = era5_data.copy()
era5_seas = era5_df[
    (era5_df["valid_month"].isin(season_months))
    & (era5_df["pcode"].isin(season_csv["adm2_pcode"].unique()))
].copy()
# sum grouping by pcode and year
era5_seas = (
    era5_seas.groupby(["pcode", "year"])["mean"]
    .sum()
    .groupby("pcode")
    .mean()
    .reset_index()
)

In [7]:
# era5_seas = era5_seas[era5_seas["mean"] > 150]
# era5_data = era5_data[era5_data["pcode"].isin(era5_seas["pcode"])]

In [8]:
seas5_data["year"] = pd.to_datetime(seas5_data["valid_date"]).dt.year
seas5_data["valid_month"] = pd.to_datetime(seas5_data["valid_date"]).dt.month

seas5_data[
    (seas5_data["pcode"] == "ET0409")
    & (seas5_data["year"] == 2006)
    & (seas5_data["valid_month"].isin(season_months))
]

,iso3,pcode,valid_date,issued_date,leadtime,adm_level,mean,median,min,max,count,sum,std,year,valid_month
62723,ETH,ET0409,2006-03-01,2005-12-01,3,2,0,0,0,0,571,175,0,2006,3
62785,ETH,ET0409,2006-03-01,2005-11-01,4,2,0,0,0,1,571,258,0,2006,3
62847,ETH,ET0409,2006-03-01,2005-10-01,5,2,1,1,0,1,571,379,0,2006,3
62909,ETH,ET0409,2006-03-01,2005-09-01,6,2,1,1,1,2,571,570,0,2006,3
62971,ETH,ET0409,2006-04-01,2005-12-01,4,2,2,2,1,3,571,"1,191",0,2006,4
63033,ETH,ET0409,2006-04-01,2005-11-01,5,2,2,2,2,3,571,"1,370",0,2006,4
63095,ETH,ET0409,2006-04-01,2005-10-01,6,2,2,2,1,3,571,"1,225",0,2006,4
63157,ETH,ET0409,2006-05-01,2005-12-01,5,2,2,2,1,3,571,"1,195",0,2006,5
63219,ETH,ET0409,2006-05-01,2005-11-01,6,2,2,2,1,3,571,"1,355",0,2006,5
109560,ETH,ET0409,2006-03-01,2006-03-01,0,2,2,2,1,2,571,911,0,2006,3


In [9]:
seas5_df = seas5_data.copy()
seas5_df["issued_date"] = pd.to_datetime(seas5_df["issued_date"])
seas5_df["valid_date"] = pd.to_datetime(seas5_df["valid_date"])

seas5_df["issued_month"] = seas5_df["issued_date"].dt.month
seas5_df["valid_month"] = seas5_df["valid_date"].dt.month
seas5_df["year"] = seas5_df["valid_date"].dt.year
seas5_df["mean"] = seas5_df["mean"] * seas5_df["valid_month"].map(days_in_month)
# seas5_df = seas5_df[seas5_df["pcode"].isin(era5_seas["pcode"])]
seas5_df = (
    seas5_df[
        (seas5_df["issued_month"] == 2) & (seas5_df["valid_month"].isin(season_months))
    ]
    .groupby(["year", "pcode"], as_index=False)["mean"]
    .sum()
)
seas5_df["rank"] = seas5_df.groupby("pcode")["mean"].rank(method="min", ascending=True)
n_years = seas5_df["year"].nunique()
all_years = seas5_df["year"].unique()
seas5_df["return_period"] = ((n_years + 1) / seas5_df["rank"]).round(1)
seas5_df = seas5_df.merge(
    eth_adm_pop[["adm2_src", "sum"]],
    left_on="pcode",
    right_on="adm2_src",
    how="left",
)
df_rp = seas5_df[seas5_df["return_period"] >= 5].copy()
drought_counts = df_rp.groupby(["year"]).size().reset_index(name="Feb (Predictive)")
# drought_counts = (
#    df_rp.groupby(["year"])["sum"].sum().reset_index(name="Feb (Predictive)")
# )

In [10]:
seas5_df["pcode"].nunique()

62

In [11]:
rangeland_wrsi = "ds-aa-eth-drought/exploration/leap/Rangeland WRSI.csv"
wrsi_csv = stratus.load_csv_from_blob(
    rangeland_wrsi, stage="dev", container_name="projects"
)
wrsi_csv = wrsi_csv.dropna(how="all")
wrsi_csv = wrsi_csv.rename(columns={"Name": "region", "Unnamed: 1": "zone"}).dropna(
    subset=["region", "zone"]
)
dk3_cols = [c for c in wrsi_csv.columns if c.endswith("_3_3")]
dk3 = wrsi_csv[["region", "zone"] + dk3_cols]

# long format
dk3_long = dk3.melt(
    id_vars=["region", "zone"], var_name="year_dekad", value_name="wrsi"
)

# extract year
dk3_long["year"] = dk3_long["year_dekad"].str.split("_").str[0].astype(int)

# drop missing WRSI
dk3_long = dk3_long.dropna(subset=["wrsi"])


def classify_wrsi(x):
    if x <= 20:
        return "Very poor"
    elif x <= 40:
        return "Poor"
    elif x <= 60:
        return "Average"
    elif x <= 80:
        return "Good"
    else:
        return "Very good"


dk3_long["wrsi_class"] = dk3_long["wrsi"].apply(classify_wrsi)
zone_counts = (
    dk3_long.groupby(["year", "wrsi_class"])
    .size()
    .reset_index(name="n_zones")
    .sort_values(["year", "wrsi_class"])
)
zones_per_year = dk3_long.groupby("year")["zone"].nunique()
class_order = ["Very poor", "Poor", "Average", "Good", "Very good"]
zone_counts_wide = (
    zone_counts.assign(
        wrsi_class=lambda d: pd.Categorical(d["wrsi_class"], class_order)
    )
    .pivot(index="year", columns="wrsi_class", values="n_zones")
    .fillna(0)
    .sort_index()
)
# total poor + very poor zones per year
mar_rangeland_poor_vpoor = (
    zone_counts_wide[["Very poor", "Poor"]]
    .sum(axis=1)
    .reset_index(name="Rangeland WRSI (Mar)")
)

In [12]:
rangeland_wrsi = "ds-aa-eth-drought/exploration/leap/Rangeland WRSI.csv"
wrsi_csv = stratus.load_csv_from_blob(
    rangeland_wrsi, stage="dev", container_name="projects"
)
wrsi_csv = wrsi_csv.dropna(how="all")
wrsi_csv = wrsi_csv.rename(columns={"Name": "region", "Unnamed: 1": "zone"}).dropna(
    subset=["region", "zone"]
)
dk3_cols = [c for c in wrsi_csv.columns if c.endswith("_4_3")]
dk3 = wrsi_csv[["region", "zone"] + dk3_cols]

# long format
dk3_long = dk3.melt(
    id_vars=["region", "zone"], var_name="year_dekad", value_name="wrsi"
)

# extract year
dk3_long["year"] = dk3_long["year_dekad"].str.split("_").str[0].astype(int)

# drop missing WRSI
dk3_long = dk3_long.dropna(subset=["wrsi"])


def classify_wrsi(x):
    if x <= 20:
        return "Very poor"
    elif x <= 40:
        return "Poor"
    elif x <= 60:
        return "Average"
    elif x <= 80:
        return "Good"
    else:
        return "Very good"


dk3_long["wrsi_class"] = dk3_long["wrsi"].apply(classify_wrsi)
zone_counts = (
    dk3_long.groupby(["year", "wrsi_class"])
    .size()
    .reset_index(name="n_zones")
    .sort_values(["year", "wrsi_class"])
)
zones_per_year = dk3_long.groupby("year")["zone"].nunique()
class_order = ["Very poor", "Poor", "Average", "Good", "Very good"]
zone_counts_wide = (
    zone_counts.assign(
        wrsi_class=lambda d: pd.Categorical(d["wrsi_class"], class_order)
    )
    .pivot(index="year", columns="wrsi_class", values="n_zones")
    .fillna(0)
    .sort_index()
)
# total poor + very poor zones per year
apr_rangeland_poor_vpoor = (
    zone_counts_wide[["Very poor", "Poor"]]
    .sum(axis=1)
    .reset_index(name="Rangeland WRSI (Apr)")
)

In [13]:
rangeland_wrsi = "ds-aa-eth-drought/exploration/leap/Rangeland WRSI.csv"
wrsi_csv = stratus.load_csv_from_blob(
    rangeland_wrsi, stage="dev", container_name="projects"
)
wrsi_csv = wrsi_csv.dropna(how="all")
wrsi_csv = wrsi_csv.rename(columns={"Name": "region", "Unnamed: 1": "zone"}).dropna(
    subset=["region", "zone"]
)
dk3_cols = [c for c in wrsi_csv.columns if c.endswith("_5_3")]
dk3 = wrsi_csv[["region", "zone"] + dk3_cols]

# long format
dk3_long = dk3.melt(
    id_vars=["region", "zone"], var_name="year_dekad", value_name="wrsi"
)

# extract year
dk3_long["year"] = dk3_long["year_dekad"].str.split("_").str[0].astype(int)

# drop missing WRSI
dk3_long = dk3_long.dropna(subset=["wrsi"])


def classify_wrsi(x):
    if x <= 20:
        return "Very poor"
    elif x <= 40:
        return "Poor"
    elif x <= 60:
        return "Average"
    elif x <= 80:
        return "Good"
    else:
        return "Very good"


dk3_long["wrsi_class"] = dk3_long["wrsi"].apply(classify_wrsi)
zone_counts = (
    dk3_long.groupby(["year", "wrsi_class"])
    .size()
    .reset_index(name="n_zones")
    .sort_values(["year", "wrsi_class"])
)
zones_per_year = dk3_long.groupby("year")["zone"].nunique()
class_order = ["Very poor", "Poor", "Average", "Good", "Very good"]
zone_counts_wide = (
    zone_counts.assign(
        wrsi_class=lambda d: pd.Categorical(d["wrsi_class"], class_order)
    )
    .pivot(index="year", columns="wrsi_class", values="n_zones")
    .fillna(0)
    .sort_index()
)
# total poor + very poor zones per year
seas_rangeland_poor_vpoor = (
    zone_counts_wide[["Very poor", "Poor"]]
    .sum(axis=1)
    .reset_index(name="Rangeland WRSI (May)")
)

In [14]:
# hindcasts by region
belg_wrsi = "ds-aa-eth-drought/exploration/leap/Final_Index_Basket_Belg.csv"
df = stratus.load_csv_from_blob(belg_wrsi, stage="dev", container_name="projects")
# --- Extract years from first row ---
years = df.iloc[0, 2:].astype(int).values

# --- Drop header row and rename columns ---
data = df.iloc[1:].copy()
data.columns = ["region", "zone"] + list(years)

# --- Melt to long format ---
long_df = data.melt(
    id_vars=["region", "zone"], var_name="year", value_name="index_value"
)

# Ensure numeric
long_df["index_value"] = pd.to_numeric(long_df["index_value"], errors="coerce")

# --- Define categories ---
bins = [0, 50, 60, 80, 90, 95, 100]
labels = [
    "Complete failure (0–50)",
    "Poor (50–60)",
    "Mediocre (60–80)",
    "Average (80–90)",
    "Good (90–95)",
    "Very good (95–100)",
]

long_df["category"] = pd.cut(
    long_df["index_value"], bins=bins, labels=labels, right=True, include_lowest=True
)

# --- Count zones per year per category ---
counts = (
    long_df.groupby(["year", "category"], observed=False)
    .size()
    .reset_index(name="n_zones")
)

counts_wide = (
    counts.pivot(index="year", columns="category", values="n_zones")
    .fillna(0)
    .astype(int)
)

# Ensure correct column order
counts_wide = counts_wide[labels]
# total poor + very poor zones per year
cropland_poor_vpoor = (
    long_df[long_df["category"].isin(["Complete failure (0–50)", "Poor (50–60)"])]
    .groupby("year")
    .size()
    .reset_index(name="Cropland basket Final WRSI")
)

In [15]:
# looking at the yield reduction
season = "Belg"  # or Belg/Meher
yield_reduction_path = (
    f"ds-aa-eth-drought/exploration/leap/Yield_Reduction_Basket_{season}.csv"
)
yield_reduction = stratus.load_csv_from_blob(
    yield_reduction_path, stage="dev", container_name="projects"
)
adm2_col = yield_reduction.iloc[1:, 1].rename("adm2_name")

# keep only valid year columns
year_cols = [
    c for c in yield_reduction.columns if c.startswith("Basket_Yield_Reduction_")
]

years = [int(c.split("_")[-1]) for c in year_cols]
df = yield_reduction[["ZONENAME"] + year_cols].copy()
df = df.rename(columns={"ZONENAME": "adm2_name"})
df.columns = ["adm2_name"] + years

# ensure numeric
df.iloc[:, 1:] = df.iloc[:, 1:].apply(pd.to_numeric, errors="coerce")
# drop col 1996
df = df.drop(columns=[1996])
# count how many adm2 have yield reduction >= 50%
yield_reduction_counts = (
    df.set_index("adm2_name")
    .ge(50)
    .sum()
    .reset_index(name="Yield Reduction ≥ 50%")
    .rename(columns={"index": "year"})
)
yield_reduction_counts.head()

,year,Yield Reduction ≥ 50%
0,2025,6
1,2024,6
2,2023,3
3,2022,16
4,2021,3


In [16]:
era_zone_year = (
    era5_data[era5_data["valid_month"].isin([3])]
    .groupby(["pcode", "year"])["mean"]
    .sum()
    .reset_index()
)

era_zone_year["rank"] = era_zone_year.groupby("pcode")["mean"].rank(
    method="min", ascending=True
)
n_years = era_zone_year["year"].nunique()
era_zone_year["return_period"] = ((n_years + 1) / era_zone_year["rank"]).round(1)

# merge population at zone level
era_zone_year = era_zone_year.merge(
    eth_adm_pop[["adm2_src", "sum"]],
    left_on="pcode",
    right_on="adm2_src",
    how="left",
)

# sum population for zones crossing 1-in-5 RP (March)
era5_zones_march_pop = (
    era_zone_year[era_zone_year["return_period"] >= 5]
    .groupby("year")  # ["sum"]
    .size()
    .reset_index(name="Affected Zones (March)")
)

In [17]:
era_zone_year = (
    era5_data[era5_data["valid_month"].isin(season_months)]
    .groupby(["pcode", "year"])["mean"]
    .sum()
    .reset_index()
)

era_zone_year["rank"] = era_zone_year.groupby("pcode")["mean"].rank(
    method="min", ascending=True
)
n_years = era_zone_year["year"].nunique()
era_zone_year["return_period"] = ((n_years + 1) / era_zone_year["rank"]).round(1)

# merge population at zone level
era_zone_year = era_zone_year.merge(
    eth_adm_pop[["adm2_src", "sum"]],
    left_on="pcode",
    right_on="adm2_src",
    how="left",
)

# sum population for zones crossing 1-in-5 RP
era5_zones_seas_pop = (
    era_zone_year[era_zone_year["return_period"] >= 5]
    .groupby("year")  # ["sum"]
    .size()
    .reset_index(name="Zones (Observed)")
)

In [18]:
# vhi and asi data
vhi_asi_data = stratus.load_csv_from_blob(
    "ds-aa-eth-drought/processed/eth_asi_mam_jjas_ond.csv",
    stage="dev",
    container_name="projects",
)

In [25]:
era5_seasonal = (
    era5_data[era5_data["valid_month"].isin(season_months)]
    .groupby(["year", "pcode", "valid_month"], as_index=False)["mean"]
    .mean()
    .groupby(["year", "pcode"], as_index=False)["mean"]
    .sum()
    .groupby("year", as_index=False)["mean"]
    .mean()
    .rename(columns={"mean": "ERA5 Seasonal Rainfall"})
    .sort_values("ERA5 Seasonal Rainfall", ascending=True)
    .reset_index(drop=True)
)
era5_seasonal = era5_seasonal.merge(drought_counts, on="year", how="left")
# era5_seasonal = era5_seasonal.merge(era5_zones_march_pop, on="year", how="left")
# era5_seasonal = era5_seasonal.merge(mar_rangeland_poor_vpoor, on="year", how="left")
# era5_seasonal = era5_seasonal.merge(apr_rangeland_poor_vpoor, on="year", how="left")
# era5_seasonal = era5_seasonal.merge(seas_rangeland_poor_vpoor, on="year", how="left")
# era5_seasonal = era5_seasonal.merge(era5_zones_seas_pop, on="year", how="left")
# era5_seasonal = era5_seasonal.merge(cropland_poor_vpoor, on="year", how="left")
# era5_seasonal = era5_seasonal.merge(
#    yield_reduction_counts, left_on="year", right_on="year", how="left"
# )
era5_seasonal = era5_seasonal.merge(
    validation_csv.drop(columns=["Season"]),
    left_on="year",
    right_on="year",
    how="left",
)
asi_cols = ["year"] + [
    c for c in vhi_asi_data.columns if c.lower().startswith("mam_asi_wavg")
]

era5_seasonal = era5_seasonal.merge(
    vhi_asi_data[asi_cols],
    on="year",
    how="left",
)
era5_seasonal = era5_seasonal.rename(
    columns=lambda c: (
        c.lower().replace("mam_", "").replace("_", " ").upper()
        if c.lower().startswith("mam_")
        else c
    )
)
era5_seasonal = era5_seasonal.rename(
    columns={
        "year": "Year",
        "ERA5 Seasonal Rainfall": "ERA5 Seasonal Rainfall (mm)",
        "Feb (Predictive)": "SEAS5 Feb Forecast (Number of Zones ≥ 5 Yr RP)",
        "Affected Zones (March)": "ERA5 March (Number of Zones ≥ 5 Yr RP)",
        "Rangeland WRSI (Mar)": "Rangeland WRSI (Mar, poor+very poor)",
        "Rangeland WRSI (Apr)": "Rangeland WRSI (Apr, poor+very poor)",
        "Rangeland WRSI (May)": "Rangeland WRSI (May, poor+very poor)",
        "Zones (Observed)": "Zones ≥ 5-Year RP (Observed)",
        "Cropland basket Final WRSI": "Cropland WRSI (Final, poor+complete failure)",
        "Yield Reduction ≥ 50%": "Zones with Yield Reduction ≥ 50%",
        # "Affected Population (April)": "ERA5 April – Population at Risk",
        "CERF Allocations": "CERF Allocations (USD)",
        "People Affected": "People Affected",
        "ASI WAVG": "ASI",
    }
)
HIGHLIGHT_RULES = {}

for col in era5_seasonal.columns:
    if col.startswith("SEAS5 Feb Forecast"):
        HIGHLIGHT_RULES[col] = (15, "#78A2D8")

    elif col.startswith("ERA5 March"):
        HIGHLIGHT_RULES[col] = (26, "#DABF86")

    elif col.startswith("Rangeland WRSI (Mar"):
        HIGHLIGHT_RULES[col] = (28, "#DABF86")

    elif col.startswith("Rangeland WRSI (Apr"):
        HIGHLIGHT_RULES[col] = (24, "#DABF86")

    elif col.startswith("Rangeland WRSI (May"):
        HIGHLIGHT_RULES[col] = (20, "#72D28F")

    elif col.startswith("Cropland WRSI"):
        HIGHLIGHT_RULES[col] = (24, "#72D28F")

    elif col.startswith("Zones ≥"):
        HIGHLIGHT_RULES[col] = (20, "#72D28F")

    elif col.startswith("Zones with Yield Reduction"):
        HIGHLIGHT_RULES[col] = (15, "#72D28F")

    elif col.startswith("ASI"):
        HIGHLIGHT_RULES[col] = (35, "#D4782C")

In [26]:
styled_table = era5_seasonal.style.format(
    {
        c: ("{:,.1f}" if c.lower().startswith(("asi", "vhi")) else "{:,.0f}")
        for c in era5_seasonal.columns
        if (pd.api.types.is_numeric_dtype(era5_seasonal[c]) and c != "Year")
    },
    na_rep="",
)


for col, (thr, color) in HIGHLIGHT_RULES.items():
    styled_table = styled_table.map(
        lambda v, t=thr, c=color: (
            f"background-color: {c};" if pd.notna(v) and v >= t else ""
        ),
        subset=[col],
    )
styled_table

,Year,ERA5 Seasonal Rainfall (mm),SEAS5 Feb Forecast (Number of Zones ≥ 5 Yr RP),CERF Allocations (USD),People Affected,ASI
0,2022,192,16,"11,999,748","24,100,000",54.5
1,2009,233,55,,"6,200,000",62.4
2,2000,241,41,,,13.2
3,2008,247,23,,"6,400,000",46.1
4,2021,276,34,"19,996,683","6,800,000",21.4
5,2011,286,,"14,598,379","4,805,679",44.4
6,2012,288,28,,"1,000,000",31.8
7,2019,290,29,"9,998,667",,20.9
8,2017,291,,"10,000,000",,26.1
9,2007,298,5,,,18.7


In [27]:
trigger_mask = pd.Series(False, index=era5_seasonal.index)

for col, (thr, _) in HIGHLIGHT_RULES.items():
    if (
        col in era5_seasonal.columns
        and not col.lower().startswith("era5")
        and not col.lower().startswith(("asi", "vhi"))
    ):
        trigger_mask |= pd.to_numeric(era5_seasonal[col], errors="coerce") >= thr


styled_table = styled_table.apply(
    lambda s: ["font-weight: bold;" if trigger_mask.loc[i] else "" for i in s.index],
    axis=0,
)

styled_table

,Year,ERA5 Seasonal Rainfall (mm),SEAS5 Feb Forecast (Number of Zones ≥ 5 Yr RP),CERF Allocations (USD),People Affected,ASI
0,2022,192,16,"11,999,748","24,100,000",54.5
1,2009,233,55,,"6,200,000",62.4
2,2000,241,41,,,13.2
3,2008,247,23,,"6,400,000",46.1
4,2021,276,34,"19,996,683","6,800,000",21.4
5,2011,286,,"14,598,379","4,805,679",44.4
6,2012,288,28,,"1,000,000",31.8
7,2019,290,29,"9,998,667",,20.9
8,2017,291,,"10,000,000",,26.1
9,2007,298,5,,,18.7
